In [ ]:
# EX 1 Creation du topic : 

In [7]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers='kafka:9092')

admin.create_topics([NewTopic(name='weather_stream', num_partitions=1, replication_factor=1)])
print("Topic crée")

print(admin.list_topics())
admin.close()


Topic crée
['weather_stream', '__consumer_offsets']


In [ ]:
# EX 1 Envoi du message :

In [8]:
import json
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers='kafka:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

producer.send('weather_stream', {"msg": "Hello Kafka"})
producer.flush()  
print("Message envoyé")


Message envoyé


In [5]:
# EX 2 : Creation du consommateur avec variable pour le topic + réception du message: 

In [4]:
from kafka import KafkaConsumer
topic = 'weather_stream'
consumer = KafkaConsumer(
    topic,
    bootstrap_servers='kafka:9092',
    auto_offset_reset='earliest',   
    consumer_timeout_ms=5000        
)
for message in consumer:
    print(message.value.decode())

print("Fin de la lecture.")


{"msg": "Hello Kafka"}
{"msg": "Hello Kafka"}
Fin de la lecture.


In [ ]:
# EX 3 : Producteur current weather qui interroge l'api toutes les 30 secondes

In [6]:
import json
import time
import requests
from kafka import KafkaProducer

CITIES = [
    {"city": "Paris", "country": "France","latitude": 48.8566, "longitude": 2.3522},
    {"city": "Seville", "country": "Spain", "latitude": 37.3891, "longitude": -5.9845},
    {"city": "Dubai", "country": "United Arab Emirates", "latitude": 25.2048,  "longitude": 55.2708},
    {"city": "Reykjavik", "country": "Iceland",    "latitude": 64.1466, "longitude": -21.9426},
    {"city": "Wellington","country": "New Zealand","latitude": -41.2865, "longitude": 174.7762},
]

URL = ("https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m,weather_code&wind_speed_unit=ms")

while True:
    for c in CITIES:
        try:
            url = URL.format(latitude=c["latitude"], longitude=c["longitude"])
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            data = response.json()
            data["city"] = c["city"]
            data["country"] = c["country"]
            producer.send('weather_stream', data)
            print(f"Envoyé : {c['city']} → {data['current']['temperature_2m']} °C")
        except Exception as e:
            print(f"Erreur pour {c['city']} : {e}")

    producer.flush()
    time.sleep(30)


Envoyé : Paris → 12.3 °C
Envoyé : Seville → 21.4 °C
Envoyé : Dubai → 30.5 °C
Envoyé : Reykjavik → 6.8 °C
Envoyé : Wellington → 15.4 °C
Envoyé : Paris → 12.3 °C
Envoyé : Seville → 21.4 °C
Envoyé : Dubai → 30.5 °C
Envoyé : Reykjavik → 6.8 °C
Envoyé : Wellington → 15.4 °C
Envoyé : Paris → 12.3 °C
Envoyé : Seville → 21.4 °C
Envoyé : Dubai → 30.5 °C
Envoyé : Reykjavik → 6.8 °C
Envoyé : Wellington → 15.4 °C


KeyboardInterrupt: 

In [ ]:
# 4 transformation des données et detections d'alertes :

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("weather_stream")
         .master("spark://spark-master:7077")
         .getOrCreate())
